# 🥔 Potato Disease Classification - MobileNetV2 (Understand Version)

This notebook is my study version of the MobileNetV2 model. It has the **same code** as the deliverable, but with **rich explanations** before each cell, simple analogies, and personal notes — to actually understand what is happening, not just run it.

---

## 🎯 What this notebook does (in one sentence)

Use **transfer learning** with **MobileNetV2** (pre-trained on ImageNet) to classify potato leaf images into 3 classes: Early Blight 🟫, Healthy 🟢, Late Blight 🟤.

## 🆚 Why a new notebook (vs the CNN one)?

Keeping the CNN baseline and the MobileNetV2 model in **separate notebooks** because:
- Side-by-side comparison is easier for the prof.
- The CNN one stays preserved as evidence/baseline.
- If MobileNetV2 has issues, the old notebook still works.

## 🗺️ Big picture roadmap

1. **Imports** — bring in MobileNetV2 and friends.
2. **Offline augmentation** — already done in the previous run, this cell will skip.
3. **Load the dataset** — read images at 224×224 (MobileNetV2 default size).
4. **Show samples** — visual sanity check.
5. **Split** — 80/10/10 with same seed=12 as the CNN, for fair comparison.
6. **Cache + prefetch** — speed up training.
7. **On-the-fly augmentation** — kept for regularization.
8. **Build the MobileNetV2 model** — frozen base + small head.
9. **Compile** — Adam, lr=1e-3.
10. **Train** — 10 epochs (less than CNN, transfer learning converges fast).
11. **Plot curves** — accuracy/loss over epochs.
12. **Sample predictions** — visual check on test images.
13. **Evaluate + save** — final test score.
14. **Confusion matrix + report** — per-class breakdown.
15. **TFLite cell** — kept commented, will run after prof's review.
16. **Q&A bank** — ready answers for the prof discussion.

## 1️⃣ Imports 📦

Two new things compared to the CNN notebook:

| New thing | What it is for |
|---|---|
| `MobileNetV2` | The pre-trained model from `tensorflow.keras.applications`. Imports the architecture and weights trained on ImageNet. |
| `preprocess_input` | A small helper function that scales pixels from [0, 255] to [-1, 1]. This is the exact range MobileNetV2 was trained on, so we must use it (not /255 like the CNN). |
| `GlobalAveragePooling2D` | Replaces `Flatten` from the CNN. Takes a 2D feature map (7×7×1280) and averages each channel into a single number → output is a 1D vector of 1280 numbers. |

> 📝 *My note (Khaled): Functional API will be used here (not Sequential) because we need to pass `training=False` to the base model. This keeps batch normalization in inference mode during our training.*

In [ ]:
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

import matplotlib.pyplot as plt
import numpy as np

print(tf.__version__)

2.19.1


## 2️⃣ Offline augmentation for the Healthy class 🩹

Same script as the CNN notebook. The Healthy folder was already augmented before (152 → 760), so the `already_augmented` check will skip the cell this time.

### 🛡️ Why we still keep this cell

Even though the augmented files exist on disk, keeping this cell makes the notebook **self-contained**: if I run it on a fresh dataset (or a different machine), it will recreate the augmented files automatically. The check protects against double-augmentation when re-running.

In [ ]:
import os

healthy_dir = "./PlantVillage/Potato___healthy"
existing = [f for f in os.listdir(healthy_dir) if f.lower().endswith(('.jpg', '.jpeg'))]
print(f"Healthy images before offline augmentation: {len(existing)}")

already_augmented = any(f.startswith("aug") for f in existing)

if not already_augmented:
    offline_aug = Sequential([
        tf.keras.layers.RandomFlip("horizontal_and_vertical"),
        tf.keras.layers.RandomRotation(0.2),
        tf.keras.layers.RandomZoom(0.1),
    ])

    N_AUG = 4
    for img_name in existing:
        img_path = os.path.join(healthy_dir, img_name)
        img = tf.keras.preprocessing.image.load_img(img_path)
        img_array = tf.keras.preprocessing.image.img_to_array(img)
        img_array = tf.expand_dims(img_array, 0)
        for i in range(N_AUG):
            augmented = offline_aug(img_array, training=True)
            new_img = tf.keras.preprocessing.image.array_to_img(augmented[0])
            new_name = f"aug{i}_{img_name}"
            new_img.save(os.path.join(healthy_dir, new_name))
    print("Offline augmentation done.")
else:
    print("Already augmented, skipping.")

after = [f for f in os.listdir(healthy_dir) if f.lower().endswith(('.jpg', '.jpeg'))]
print(f"Healthy images after offline augmentation: {len(after)}")

Healthy images before offline augmentation: 760
Already augmented, skipping.
Healthy images after offline augmentation: 760


## 3️⃣ Load the dataset 📂

### 🆚 Two changes vs the CNN notebook

**1) `IMAGE_SIZE = 224` (not 256)**
MobileNetV2 was trained on **224×224** images on ImageNet. If I feed it a different size, the pre-trained weights stop matching the input shape they expect, and accuracy drops. So I match what MobileNetV2 expects.

**2) `EPOCHS = 10` (not 20)**
Transfer learning converges much faster than training from scratch. With a frozen pre-trained base, the model is already at ~91% accuracy after epoch 1. By epoch 5 it's around 99%. Going to 20 epochs would just waste time.

Everything else (batch size, channels, the explicit `class_names` list) stays the same as the CNN notebook for fair comparison.

In [ ]:
IMAGE_SIZE = 224   # MobileNetV2 default input
BATCH_SIZE = 32
CHANNELS = 3
EPOCHS = 10

dataset = tf.keras.preprocessing.image_dataset_from_directory(
    "./PlantVillage",
    class_names=["Potato___Early_blight", "Potato___healthy", "Potato___Late_blight"],
    shuffle=True,
    image_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE
)

class_names = dataset.class_names
print(class_names)

Found 2760 files belonging to 3 classes.
['Potato___Early_blight', 'Potato___healthy', 'Potato___Late_blight']


## 4️⃣ Show sample images 👀

Visual sanity check — same as in the CNN notebook. We grab 1 batch (32 images) and display the first 12 in a 3×4 grid with their class labels.

**What to look for:**
- Real leaf images, not corrupted.
- Labels match the visual content (a Healthy leaf is labeled Healthy).

In [ ]:
plt.figure(figsize=(10, 10))

for image_batch, label_batch in dataset.take(1):
    for i in range(12):
        ax = plt.subplot(3, 4, i + 1)
        plt.imshow(image_batch[i].numpy().astype("uint8"))
        plt.title(class_names[label_batch[i]])
        plt.axis("off")

> 🖼️ **Plot output (not shown in this Understand notebook):** 3×4 grid of sample leaf images with class labels as titles. All labels look correct and images are clean.

## 5️⃣ Split into train / validation / test 🍰

Same function as in the CNN notebook, with the **same seed=12**, so:

✅ The MobileNetV2 model and the CNN baseline are trained, validated, and tested on **exactly the same images**. This makes the comparison fair.

### 🧮 Numbers in our case

- Total batches: 2,760 / 32 ≈ **87 batches**.
- Train: ~69 batches.
- Val: ~8 batches.
- Test: ~10 batches × 32 = **320 test images**.

> 📝 *My note (Khaled): I noticed before that 320 test images is bigger than the 276 I expected (10% × 2,760 = 276). The reason is that the split is done by **batches**, not by individual images. After the integer rounding, train+val=77 batches, so 87-77=10 batches go to test, which gives 10×32=320 images.*

In [ ]:
def get_dataset_partitions(ds, train_split=0.8, val_split=0.1, shuffle=True, shuffle_size=10000):
    ds_size = len(ds)

    if shuffle:
        ds = ds.shuffle(shuffle_size, seed=12)

    train_size = int(train_split * ds_size)
    val_size = int(val_split * ds_size)

    train_ds = ds.take(train_size)
    val_ds = ds.skip(train_size).take(val_size)
    test_ds = ds.skip(train_size).skip(val_size)

    return train_ds, val_ds, test_ds

train_ds, val_ds, test_ds = get_dataset_partitions(dataset)

print(len(train_ds), len(val_ds), len(test_ds))

69 8 10


## 6️⃣ Cache + prefetch ⚡

Same as before. Cache decoded images in memory after the 1st epoch, shuffle the order each epoch, and prefetch the next batch while the current one trains.

Saves time across the 10 epochs.

In [ ]:
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=tf.data.AUTOTUNE)
val_ds   = val_ds.cache().shuffle(1000).prefetch(buffer_size=tf.data.AUTOTUNE)
test_ds  = test_ds.cache().shuffle(1000).prefetch(buffer_size=tf.data.AUTOTUNE)

## 7️⃣ On-the-fly data augmentation 🎲

### 🆚 Different from the CNN notebook here

In the CNN notebook we had **two** Sequential blocks: `resize_and_rescale` (with `Rescaling(1/255)`) and `data_augmentation`.

Here we only need the **augmentation block**. We do not need a Rescaling layer because:
- The resize part is already done by `image_dataset_from_directory(image_size=(224, 224))`.
- The rescaling will be done inside the model by `preprocess_input` (next cell), which scales to [-1, 1] instead of [0, 1].

### 🤔 Why two stages of augmentation?

| Stage | When | Where saved | Purpose |
|---|---|---|---|
| **Offline** (cell 2) | Before training, once | On disk | Fix class imbalance (152 → 760) |
| **On-the-fly** (here) | At every epoch | Only in memory | Regularization (model can't memorize fixed pixels) |

Both run for all 3 classes during training, to keep the augmentation policy uniform.

In [ ]:
data_augmentation = Sequential([
    tf.keras.layers.RandomFlip("horizontal_and_vertical"),
    tf.keras.layers.RandomRotation(0.2),
])

## 8️⃣ Build the MobileNetV2 model 🏗️

### 🧠 What is transfer learning?

Take a model that was already trained on millions of images (ImageNet has 1.2 million images, 1000 classes) and **re-use its learned features** for our small problem. The early layers of any image model learn general things (edges, colors, textures) that work for any image, not just ImageNet, so we save them.

### 🧠 What is "frozen base"?

Setting `base_model.trainable = False` tells Keras: *don't update these weights during training*. The MobileNetV2 base stays exactly as ImageNet trained it. Only our small classifier head on top is trained on potato leaves.

### 📚 Layer-by-layer

| # | Layer | What it does |
|---|---|---|
| 1 | `Input(shape=(224,224,3))` | The image enters here. |
| 2 | `data_augmentation` | Random flip + rotation (training only). |
| 3 | `preprocess_input` | Scales pixels from [0,255] to [-1,1]. |
| 4 | MobileNetV2 base (FROZEN) | 53 conv layers from ImageNet. Output: 7×7×1280 feature map. |
| 5 | `GlobalAveragePooling2D` | Averages each channel → 1D vector of 1280 numbers. Replaces Flatten. |
| 6 | `Dense(64) + ReLU` | Small mixing layer. |
| 7 | `Dense(3) + Softmax` | Output: 3 probabilities for the 3 classes. |

### 🤔 Why Functional API (not Sequential)?

Because we need `base_model(x, training=False)` to keep batch normalization layers in **inference mode**, even during training. With Sequential we cannot pass `training=False` to a single layer in the middle.

### 🤔 Why `GlobalAveragePooling2D` instead of `Flatten`?

If I used Flatten on the 7×7×1280 output, I would get a vector of 7×7×1280 = **62,720 numbers**, and the next Dense(64) would have 62,720 × 64 ≈ **4 million** parameters. Way too much for our small dataset → overfitting.

GlobalAveragePooling2D averages across the spatial dimensions and gives just **1280 numbers**, so Dense(64) only has 1280 × 64 = ~82k parameters. Much safer.

> 📝 *My note (Khaled): the warning about `input_shape` in newer Keras versions is just stylistic — the model still works fine. Same warning we saw in the CNN notebook.*

In [ ]:
input_shape = (IMAGE_SIZE, IMAGE_SIZE, CHANNELS)
n_classes = 3

base_model = MobileNetV2(input_shape=input_shape, include_top=False, weights='imagenet')
base_model.trainable = False

inputs = tf.keras.Input(shape=input_shape)
x = data_augmentation(inputs)
x = preprocess_input(x)
x = base_model(x, training=False)
x = GlobalAveragePooling2D()(x)
x = Dense(64, activation='relu')(x)
outputs = Dense(n_classes, activation='softmax')(x)

model = tf.keras.Model(inputs, outputs)
model.summary()

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 224, 224, 3)]     0         
 sequential (Sequential)     (None, 224, 224, 3)       0         
 tf.math.truediv (TFOpLambda)(None, 224, 224, 3)       0         
 tf.math.subtract (TFOpLambd)(None, 224, 224, 3)       0         
 mobilenetv2_1.00_224 (Funct)(None, 7, 7, 1280)        2257984   
 global_average_pooling2d    (None, 1280)              0         
 dense (Dense)               (None, 64)                81984     
 dense_1 (Dense)             (None, 3)                 195       
Total params: 2,340,163
Trainable params: 82,179
Non-trainable params: 2,257,984
_________________________________________________________________


## 9️⃣ Compile ⚙️

Same idea as before: tell Keras which optimizer, loss, and metric to use.

**One small change** vs the CNN: we explicitly set `learning_rate=1e-3`. (In the CNN notebook we used `optimizer='adam'` which uses the same default lr=1e-3 anyway, but writing it explicitly makes the choice obvious for the prof.)

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

## 🔟 Train the model 🏋️

Same call as before: `model.fit(...)`.

### ⏱️ Expected time

Around 13 minutes on CPU for 10 epochs. Faster than the CNN baseline (24 min for 20 epochs) because:
- Less epochs (10 vs 20).
- Less trainable parameters (82,179 vs 183,747) → less weights to update each step.

### 📊 What to expect

Transfer learning starts very strong:
- Epoch 1 already at ~92% accuracy.
- By epoch 5: ~99%.
- Final epoch: ~99.5%.

Compare with the CNN that started at ~56% and reached 97% only at the end.

In [ ]:
history = model.fit(
    train_ds,
    batch_size=BATCH_SIZE,
    validation_data=val_ds,
    verbose=1,
    epochs=EPOCHS
)

Epoch 1/10
69/69 [==============================] - 73s 1s/step - accuracy: 0.9185 - loss: 0.2116 - val_accuracy: 0.9766 - val_loss: 0.0948
Epoch 2/10
69/69 [==============================] - 71s 1s/step - accuracy: 0.9783 - loss: 0.0689 - val_accuracy: 0.9883 - val_loss: 0.0421
Epoch 3/10
69/69 [==============================] - 71s 1s/step - accuracy: 0.9891 - loss: 0.0398 - val_accuracy: 0.9909 - val_loss: 0.0312
...
Epoch 8/10
69/69 [==============================] - 71s 1s/step - accuracy: 0.9952 - loss: 0.0156 - val_accuracy: 0.9974 - val_loss: 0.0098
Epoch 9/10
69/69 [==============================] - 71s 1s/step - accuracy: 0.9961 - loss: 0.0138 - val_accuracy: 0.9974 - val_loss: 0.0089
Epoch 10/10
69/69 [==============================] - 71s 1s/step - accuracy: 0.9971 - loss: 0.0125 - val_accuracy: 0.9987 - val_loss: 0.0078


## 1️⃣1️⃣ Plot accuracy and loss curves 📈

Same plotting code as the CNN notebook. Two side-by-side plots showing accuracy and loss for both training and validation across the 10 epochs.

### 🩺 What to look for

- Training and validation curves stay close to each other → no overfitting.
- Both curves rise very fast in the first few epochs, then flatten near the top → transfer learning behaves as expected.

In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

loss = history.history['loss']
val_loss = history.history['val_loss']

epochs_range = range(EPOCHS)

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy')
plt.plot(epochs_range, val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.title('Training and Validation Accuracy')

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Training Loss')
plt.plot(epochs_range, val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.title('Training and Validation Loss')

plt.show()

> 🖼️ **Plot output (not shown in this Understand notebook):** Two side-by-side curves. Left = accuracy (both rise from ~92% at epoch 1 to ~99.7% at epoch 10). Right = loss (both drop from ~0.2 to ~0.01). Training and validation curves stay tight together — no overfitting visible.

## 1️⃣2️⃣ Sample predictions 🔍

Same idea as the CNN notebook. Pick 9 random test images, run them through the model, show actual class + predicted class + confidence %.

Just a visual sanity check — the official evaluation comes from `model.evaluate(...)` in the next cell.

In [ ]:
def predict(model, img):
    img_array = tf.keras.preprocessing.image.img_to_array(img)
    img_array = tf.expand_dims(img_array, 0)

    predictions = model.predict(img_array, verbose=0)
    predicted_class = class_names[np.argmax(predictions[0])]
    confidence = round(100 * (np.max(predictions[0])), 2)
    return predicted_class, confidence

plt.figure(figsize=(15, 15))
for images, labels in test_ds.take(1):
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))

        predicted_class, confidence = predict(model, images[i].numpy())
        actual_class = class_names[labels[i]]

        plt.title(f"Actual: {actual_class}\nPredicted: {predicted_class}\nConfidence: {confidence}%")
        plt.axis("off")

> 🖼️ **Plot output (not shown in this Understand notebook):** 3×3 grid of test images. All 9 predictions correct. Confidence % is above 99% for all of them — the model is very confident.

## 1️⃣3️⃣ Evaluate on test set + save the model 💾

### 🎯 Final test

`model.evaluate(test_ds)` runs the model on the 320 test images (which the model never saw during training) and returns:
- `loss` — sparse categorical cross-entropy on test set.
- `accuracy` — % of correct predictions.

### 💾 Save

Saved as `potato_model_mobilenetv2.h5`. Different filename from the CNN model so both are preserved.

> 📝 *My note (Khaled): Keras now shows a warning that .h5 is the legacy format and prefers .keras or SavedModel. For now I keep .h5 because that's what we always used; the warning is just informational, not an error.*

In [ ]:
scores = model.evaluate(test_ds)
print(scores)

model.save("potato_model_mobilenetv2.h5")

10/10 [==============================] - 5s 537ms/step - accuracy: 1.0000 - loss: 0.0080
[0.008018075488507748, 1.0]


## 1️⃣4️⃣ Confusion matrix + classification report 🔬

Same as the CNN notebook. The single accuracy number doesn't tell the full story — the confusion matrix shows where mistakes happen, and the classification report gives precision/recall/F1 per class.

### 📚 Quick reminder

- **Confusion matrix** rows = actual classes, columns = predicted classes. Diagonal = correct.
- **Precision** of class X = of all the times the model said "X", how many were really X?
- **Recall** of class X = of all the actual X images, how many did the model catch?
- **F1-score** = balance between precision and recall.
- **Support** = how many actual images of that class were in the test set.

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

y_true = []
y_pred = []
for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    y_pred.extend(np.argmax(preds, axis=1))
    y_true.extend(labels.numpy())

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix - Test Set')
plt.show()

print(classification_report(y_true, y_pred, target_names=class_names))

                       precision    recall  f1-score   support

Potato___Early_blight       1.00      1.00      1.00       119
     Potato___healthy       1.00      1.00      1.00        83
 Potato___Late_blight       1.00      1.00      1.00       118

             accuracy                           1.00       320
            macro avg       1.00      1.00      1.00       320
         weighted avg       1.00      1.00      1.00       320


> 🖼️ **Plot output (not shown in this Understand notebook):** Confusion matrix heatmap. All 320 test images on the diagonal: Early_blight 119/119, Healthy 83/83, Late_blight 118/118. Off-diagonal cells are all 0 (zero mistakes). Looks perfect — but see the honest note below.

## ⚠️ Honest note on the 100% test accuracy

The 100% result looked too perfect to me, so I checked again to understand why. I think the main reason is related to **how the offline augmentation interacts with the train/test split**:

### 🐛 Possible data leak

1. The 4 augmented copies of each Healthy image are saved in **the same Healthy folder** as the originals (with names like `aug0_xxx.jpg`, `aug1_xxx.jpg`, …).
2. When `image_dataset_from_directory` shuffles and splits the data, it treats each file as a separate independent image.
3. So one Healthy original leaf and its augmented copy can end up in **different splits** (one in train, one in test).
4. In that case the model sees the same leaf twice (just rotated or flipped), once in training and once in test → **indirect data leak** → test accuracy looks higher than it really is.

### 💪 Plus: MobileNetV2 is genuinely strong

Even without the leak, the result would probably still be very high (maybe 97-99%) because:
- MobileNetV2 was pre-trained on millions of ImageNet images and already "knows" how to extract features from natural images.
- The 3 potato classes are visually quite different from each other (Early Blight has small dark spots, Late Blight has large irregular brown areas, Healthy is uniform green).

### 🛠️ Plan to fix in the next iteration

Do the offline augmentation **only on the training split, after the split**, instead of generating files in the folder before splitting. This way, no augmented copy of any image can leak into the validation or test split. The reported test accuracy will then be more honest.

> 📝 *My note (Khaled): I'll bring this up myself with the prof. Better to flag it than to let him discover it. Shows critical thinking.*

## 1️⃣5️⃣ Convert to TFLite (commented out for now)

We will run this **after Prof. Sheta's review**. The conversion takes about 2 minutes and produces a `.tflite` file that my partner can drop into the Flutter app.

### 🤔 Why wait?

If the prof asks for changes (different head, unfreeze last layers, fix the data leak issue), the model will need to be re-trained anyway. Converting now and handing off to the partner would be wasted work — better to wait for the green light.

### 🧠 What `Optimize.DEFAULT` does

It applies **post-training quantization** automatically. Roughly: it converts most weights from float32 (4 bytes per number) to int8 (1 byte per number). This makes the .tflite file ~4× smaller and faster on mobile, with usually less than 1% accuracy drop.

In [ ]:
# converter = tf.lite.TFLiteConverter.from_keras_model(model)
# converter.optimizations = [tf.lite.Optimize.DEFAULT]
# tflite_model = converter.convert()
#
# with open("potato_model_mobilenetv2.tflite", "wb") as f:
#     f.write(tflite_model)
#
# print("TFLite saved.")

## 🎯 Results summary

| Metric | CNN baseline | MobileNetV2 |
|---|---|---|
| Test accuracy | 94.57% | **100%** ⚠️ |
| Test loss | 0.1289 | 0.0080 |
| Trainable parameters | 183,747 | **82,179** (less!) |
| Training time | ~24 min | ~13 min |
| Epochs | 20 | 10 |

### ⚠️ Reading the 100% honestly

As explained in the honest note above, the 100% is probably inflated by the augmentation/split issue. The real accuracy on completely new leaves is most likely 97-99%, which is still a clear improvement over the CNN baseline.

### 🚀 Next step

After Prof. Sheta's review:
1. Fix the offline-aug + split issue (do augmentation only on training split).
2. Re-train and get the more honest accuracy.
3. Convert to .tflite and hand over to my partner for the Flutter app.

# ❓ Q&A Bank for Prof. Discussion

Below are the questions I think Prof. Sheta is likely to ask, with my prepared answers. Going through these before the meeting so I can answer on the spot without hesitating.

---

## 🅰️ Group A — About transfer learning and MobileNetV2

### Q1: What is transfer learning and why did you use it?

Transfer learning is when I take a model that was already trained on a huge dataset (ImageNet, 1.2M images), keep the learned weights, and only train a small classifier on top of it for my own problem. I used it because:
- The early layers of any image model learn general features (edges, textures, colors) that are useful for any image, not just ImageNet.
- My dataset is small (2,760 images). Training a big model from scratch would overfit fast. Transfer learning lets me "borrow" the knowledge from ImageNet.
- Training is much faster — no need to learn the conv filters from zero.

### Q2: Why did you choose MobileNetV2 specifically?

Three reasons:
1. It is **designed for mobile devices** — small file size (~14 MB), low memory, fast inference. This matches my final goal (a phone app).
2. It has good accuracy/cost trade-off. The architecture uses **depthwise separable convolutions** which are much cheaper than normal convolutions but still strong.
3. It is well-supported in TensorFlow Lite, with a smooth conversion path.

### Q3: What does "frozen base" mean? Why did you freeze it?

Setting `base_model.trainable = False` tells Keras: don't update these weights during training. So the MobileNetV2 base stays exactly as ImageNet trained it. Only my small classifier head (Dense 64 → Dense 3) is updated.

I froze it because:
- My dataset is small (only ~2,200 training images). If I unfroze the base, the model would have 2.3M trainable parameters and would overfit.
- The pre-trained features are already good. Re-training them on small data could actually make them worse.
- Training is much faster with only 82k trainable parameters.

### Q4: Could you fine-tune later? How?

Yes. The standard approach is two-stage:
1. **Stage 1 (what I did):** freeze the base, train only the head until it converges.
2. **Stage 2 (fine-tuning):** unfreeze the **top few layers** of MobileNetV2 (not all), use a much smaller learning rate (like 1e-5 instead of 1e-3), and train for a few more epochs.

I didn't do stage 2 yet because the model is already at very high accuracy and I want to first fix the data leak issue. Fine-tuning is something I can do later if needed.

### Q5: Why GlobalAveragePooling2D instead of Flatten?

If I used Flatten on the 7×7×1280 output, I'd get a vector of 62,720 numbers, and the next Dense(64) would have 4 million parameters. That's huge for our small dataset → overfitting risk and slower training.

GlobalAveragePooling2D averages each channel and gives just 1280 numbers, so Dense(64) only has ~82k parameters. Much safer for transfer learning.

## 🅱️ Group B — About the data and pre-processing

### Q6: Why 224×224 image size and not 256?

MobileNetV2 was pre-trained on **224×224** images on ImageNet. Its weights were learned for that input size. If I feed it a different size, the spatial dimensions in the feature maps would be different, and the pre-trained weights would not match what they expect → accuracy drops.

Same idea as: a person who learned to read at one font size can adapt, but if I suddenly change the font and layout completely they need to re-learn.

### Q7: Why preprocess_input instead of dividing by 255?

MobileNetV2 was trained with pixel values in the range **[-1, 1]** (not [0, 1]). The `preprocess_input` function from `tensorflow.keras.applications.mobilenet_v2` does this exact scaling.

If I use `Rescaling(1/255)` like in the CNN baseline, the pixels would be in [0, 1] which is the wrong range for MobileNetV2 → again the pre-trained weights don't match what they expect → accuracy drops.

### Q8: Why did you keep the same offline augmentation (152 → 760)?

Two reasons:
1. **Fairness:** I want to compare CNN vs MobileNetV2 on the **same data**. Changing the dataset between the two experiments would make the comparison unfair.
2. **The class imbalance issue is independent of the model.** The Healthy class has only 152 images regardless of which model I use, so the offline augmentation is still needed.

### Q9: How did you ensure the comparison with the CNN is fair?

Same dataset (2,760 images after offline aug), same train/val/test split with **the same random seed (12)**, same offline augmentation step (152 → 760 for Healthy). So both models are trained, validated, and tested on exactly the same images.

The only thing I changed is the model itself (custom CNN → MobileNetV2 transfer learning) and the things that depend on the model (image size 256 → 224, preprocessing /255 → preprocess_input, epochs 20 → 10).

## 🆎 Group C — About the 100% result and data leak

### Q10: Are you sure the 100% result is real?

Honest answer: **probably not 100% real.** I think there is an indirect data leak from how the offline augmentation interacts with the train/test split:

- The augmented Healthy copies are saved in the same folder as the originals.
- When `image_dataset_from_directory` shuffles and splits, an original and its augmented copy can end up in different splits.
- So the model can see the same leaf twice (just rotated/flipped), once in training and once in test.

MobileNetV2 is also genuinely very strong on this kind of task, so even without the leak the accuracy would be very high (maybe 97-99%).

### Q11: How do you plan to fix this?

Restructure the pipeline:
1. First do the train/val/test split on the original 2,152 images (no augmentation yet).
2. Then apply the offline augmentation **only on the training split**.
3. Validation and test sets stay untouched, with only original images.

This guarantees no augmented copy can leak into val or test, so the reported accuracy will be honest.

### Q12: Why didn't you catch this earlier?

Honestly, I didn't think about it when I wrote the offline augmentation cell. The simpler approach (saving aug files in the same folder before splitting) felt natural at the time, and the CNN baseline didn't reach 100% so I didn't notice anything wrong. Only when MobileNetV2 hit 100% I went back to check why.

Lesson learned: **suspiciously perfect results are usually a sign of data leak somewhere**, not of a perfect model.

## 🅳 Group D — About training and evaluation

### Q13: Why only 10 epochs and not more?

Transfer learning with a frozen base converges very fast:
- Epoch 1: already at ~92% accuracy.
- Epoch 5: ~99%.
- Epoch 10: ~99.7%.

Going to 20 or 50 epochs would just waste time without improving the model further. The validation curve also flattens after epoch 5, which confirms there's nothing more to learn.

### Q14: Why Adam with learning rate 1e-3?

1e-3 is the **default Adam learning rate** and it works well for training a small head on top of a frozen base. The head has only 82k parameters and starts from random weights, so it can absorb a relatively big learning rate.

If I were fine-tuning the base layers later (stage 2), I would use a much smaller learning rate (like 1e-5) to avoid destroying the pre-trained features.

### Q15: Why sparse_categorical_crossentropy?

Because my labels are **integers** (0 = Early_blight, 1 = healthy, 2 = Late_blight), not one-hot vectors like [0, 1, 0]. "Sparse" in the name means it accepts integer labels directly without needing to one-hot encode them first.

If I had used `categorical_crossentropy`, I would have had to convert the labels to one-hot before training. Same result, just one more step.

### Q16: What does the confusion matrix tell you?

It shows for each class, how many images were predicted as each class. The diagonal is correct, off-diagonal is mistakes.

In our case all values are on the diagonal (119, 83, 118), which means zero mistakes. **But** as I explained, this is probably inflated by the data leak.

It's still useful because if there were mistakes, the matrix would show **which classes are confused with which** — for example, Early_blight vs Late_blight is the typical confusion since they both produce dark spots.

## 🅴 Group E — About the next steps

### Q17: What is the next step after this notebook?

Three things in order:
1. **Fix the data leak issue** (move the offline augmentation to after the split).
2. **Re-train and report the more honest accuracy** (probably 97-99%).
3. **Convert the approved model to TFLite** and hand it over to my partner for the Flutter app.

### Q18: Why didn't you do the TFLite conversion in this notebook?

Two reasons:
1. I want the prof to review the model first. If he asks for changes, the .tflite file would need to be regenerated.
2. Once given the green light, the conversion is fast (~2 minutes). No time saved by doing it now.

I kept the conversion code in the notebook but commented out, with a note saying it will run after the review. So everything is ready, just not committed yet.

### Q19: How will you handover to your partner?

Three files:
1. The `.tflite` model file (~13 MB).
2. A `class_names.json` file with the class order (Early_blight=0, healthy=1, Late_blight=2).
3. A short handoff document explaining the input shape (224×224×3), the normalization (pixel/127.5 − 1.0), and the output format (3 probabilities, take argmax).

### Q20: If accuracy on the new (honest) test is much lower, what would you do?

If it drops to something like 85% (much lower than expected), I would:
1. Look at the confusion matrix to see which classes are confused.
2. Try **fine-tuning** (unfreeze the top few layers of MobileNetV2 with a small learning rate).
3. Try **stronger augmentation** (zoom, brightness, contrast) to make the model more robust.
4. If the confusion is between Early_blight and Late_blight, I might also explore other architectures (EfficientNet, ResNet) to see if any of them captures the difference better.

But I expect the honest accuracy to stay above 95%, because the visual difference between the 3 classes is genuinely big.

---

## ✅ Final checklist before the prof meeting

- [ ] Re-read the Q&A bank above (all 20 questions).
- [ ] Review the data leak explanation — be ready to bring it up myself.
- [ ] Have the comparison table CNN vs MobileNetV2 ready (numbers + why).
- [ ] Have the next-step plan ready (fix leak → re-train → TFLite → handover).
- [ ] Prepare 1-2 questions for the prof (about fine-tuning, about the offline aug fix, about timing for Grad II).

Bring laptop with the notebook open and the report printed.